<div style="background-color: #f8fafc; border-left: 6px solid #0284c7; padding: 20px 24px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.06); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
  <span style="font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 1.5px; color: #0284c7;">AIML Honors &bull; Lab Assignment 02</span>
  <h1 style="margin: 8px 0 12px 0; color: #0f172a; font-size: 24px; font-weight: 700;">Employee Record Management with Flat CSV Storage</h1>
  <div style="font-size: 14px; color: #475569; line-height: 1.6;">
    <strong>Student Name:</strong> Pranjal Khadke &nbsp;|&nbsp; 
    <strong>Roll No:</strong> 10 &nbsp;|&nbsp; 
    <strong>Email:</strong> <a href="mailto:Pranjalckhadke@gmail.com" style="color: #0284c7; text-decoration: none;">Pranjalckhadke@gmail.com</a> &nbsp;|&nbsp; 
    <strong>GitHub:</strong> <a href="https://github.com/PranjalKhadke" style="color: #0284c7; text-decoration: none;">github.com/PranjalKhadke</a><br>
    <strong>Course:</strong> AIML Honors &bull; Sem III &nbsp;|&nbsp; 
    <strong>Topic:</strong> File-Based Persistence & CRUD via Standard CSV Library
  </div>
</div>

### 1. Aim & Theoretical Overview

**Aim:** To implement a persistent, file-based CRUD (Create, Read, Update, Delete) record management system in Python using the built-in `csv` module, enforcing primary key uniqueness, data integrity validation, and aligned tabular rendering.

#### Core Mechanics of Flat CSV Storage:
- **Streaming Parser (`csv.reader`):** Reads tabular delimiter-separated text without requiring heavy third-party dependencies.
- **Atomic File Writing (`csv.writer`):** Modifying middle records in sequential text requires reading into memory, altering data structures, and committing back to disk.
- **Primary Key Constraint Simulation:** Before performing disk appends, the system validates that `emp_id` does not collide with existing row headers ($O(N)$ linear validation).

#### Data Dictionary:
| Column Name | Data Type | Constraint | Description |
| :--- | :--- | :--- | :--- |
| `emp_id` | Integer / String | **Primary Key** | Unique identifier for each employee |
| `name` | String | NOT NULL | Full name of the employee |
| `dept` | String | NOT NULL | Department designation |
| `salary` | Float | Salary $\ge 0$ | Monthly base compensation in INR |


In [1]:
import csv
import os

CSV_FILE = "employee.csv"
HEADERS = ["emp_id", "name", "dept", "salary"]

def init_file(filepath=CSV_FILE):
    """Initializes the CSV repository with predefined headers."""
    with open(filepath, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADERS)
    print(f"[SYSTEM] Initialized secure CSV repository at '{filepath}'.")

init_file()

[SYSTEM] Initialized secure CSV repository at 'employee.csv'.


In [2]:
def get_all_rows(filepath=CSV_FILE):
    if not os.path.exists(filepath):
        return []
    with open(filepath, mode="r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        rows = list(reader)
    return rows

def add_employee(emp_id, name, dept, salary, filepath=CSV_FILE):
    if not os.path.exists(filepath):
        init_file(filepath)
    rows = get_all_rows(filepath)
    for r in rows[1:]:
        if r and r[0] == str(emp_id):
            print(f"  [ERROR] Primary Key Collision: Employee ID {emp_id} already exists.")
            return False
    try:
        sal_val = float(salary)
        if sal_val < 0:
            print("  [ERROR] Salary value cannot be negative.")
            return False
    except ValueError:
        print("  [ERROR] Salary must be numeric.")
        return False
    with open(filepath, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([str(emp_id), str(name).strip(), str(dept).strip(), f"{sal_val:.2f}"])
    print(f"  [SUCCESS] Enrolled employee: {name} (ID: {emp_id}, Dept: {dept}, Salary: Rs. {sal_val:,.2f})")
    return True

def list_employees(filepath=CSV_FILE):
    rows = get_all_rows(filepath)
    if len(rows) <= 1:
        print("\n  [INFO] Employee repository is currently empty.\n")
        return
    print("\n+--------+----------------------+------------------+----------------+")
    print("| Emp ID | Employee Name        | Department       | Salary (INR)   |")
    print("+--------+----------------------+------------------+----------------+")
    for r in rows[1:]:
        if len(r) >= 4:
            sal_str = f"{float(r[3]):,.2f}"
            print(f"| {r[0]:<6} | {r[1]:<20} | {r[2]:<16} | Rs. {sal_str:>10} |")
    print("+--------+----------------------+------------------+----------------+\n")

def find_employee(emp_id, filepath=CSV_FILE):
    rows = get_all_rows(filepath)
    for r in rows[1:]:
        if r and r[0] == str(emp_id):
            sal_formatted = f"{float(r[3]):,.2f}"
            print(f"  [MATCH] ID: {r[0]} | Name: {r[1]} | Department: {r[2]} | Salary: Rs. {sal_formatted}")
            return r
    print(f"  [NOT FOUND] No employee matches ID: {emp_id}.")
    return None

def update_employee(emp_id, new_name=None, new_dept=None, new_salary=None, filepath=CSV_FILE):
    rows = get_all_rows(filepath)
    if len(rows) <= 1:
        return False
    updated = False
    for r in rows[1:]:
        if r and r[0] == str(emp_id):
            if new_name is not None and new_name.strip(): r[1] = new_name.strip()
            if new_dept is not None and new_dept.strip(): r[2] = new_dept.strip()
            if new_salary is not None: r[3] = f"{float(new_salary):.2f}"
            updated = True
            break
    if updated:
        with open(filepath, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerows(rows)
        print(f"  [SUCCESS] Successfully modified record for Employee ID {emp_id}.")
        return True
    print(f"  [NOT FOUND] Unable to update: Employee ID {emp_id} does not exist.")
    return False

def remove_employee(emp_id, filepath=CSV_FILE):
    rows = get_all_rows(filepath)
    if len(rows) <= 1: return False
    header, records = rows[0], rows[1:]
    new_records = [r for r in records if r and r[0] != str(emp_id)]
    if len(new_records) == len(records):
        print(f"  [NOT FOUND] Unable to delete: Employee ID {emp_id} does not exist.")
        return False
    with open(filepath, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(new_records)
    print(f"  [SUCCESS] Successfully expunged Employee ID {emp_id} from database.")
    return True

In [3]:
# Enrolling multiple initial employee records
add_employee(101, "Pranjal Khadke", "Engineering", 75000)
add_employee(102, "Rohan Sharma", "Data Science", 82000)
add_employee(103, "Ananya Sen", "Design", 65000)
add_employee(104, "Vikram Patel", "Operations", 58000)

  [SUCCESS] Enrolled employee: Pranjal Khadke (ID: 101, Dept: Engineering, Salary: Rs. 75,000.00)
  [SUCCESS] Enrolled employee: Rohan Sharma (ID: 102, Dept: Data Science, Salary: Rs. 82,000.00)
  [SUCCESS] Enrolled employee: Ananya Sen (ID: 103, Dept: Design, Salary: Rs. 65,000.00)
  [SUCCESS] Enrolled employee: Vikram Patel (ID: 104, Dept: Operations, Salary: Rs. 58,000.00)


In [4]:
# Displaying formatted employee roster
list_employees()


+--------+----------------------+------------------+----------------+
| Emp ID | Employee Name        | Department       | Salary (INR)   |
+--------+----------------------+------------------+----------------+
| 101    | Pranjal Khadke       | Engineering      | Rs.  75,000.00 |
| 102    | Rohan Sharma         | Data Science     | Rs.  82,000.00 |
| 103    | Ananya Sen           | Design           | Rs.  65,000.00 |
| 104    | Vikram Patel         | Operations       | Rs.  58,000.00 |
+--------+----------------------+------------------+----------------+



In [5]:
# Querying an employee by primary key (ID: 102)
find_employee(102)

  [MATCH] ID: 102 | Name: Rohan Sharma | Department: Data Science | Salary: Rs. 82,000.00


In [6]:
# Modifying department and salary for Employee ID 104
update_employee(104, new_dept="Logistics", new_salary=63500)
list_employees()

  [SUCCESS] Successfully modified record for Employee ID 104.

+--------+----------------------+------------------+----------------+
| Emp ID | Employee Name        | Department       | Salary (INR)   |
+--------+----------------------+------------------+----------------+
| 101    | Pranjal Khadke       | Engineering      | Rs.  75,000.00 |
| 102    | Rohan Sharma         | Data Science     | Rs.  82,000.00 |
| 103    | Ananya Sen           | Design           | Rs.  65,000.00 |
| 104    | Vikram Patel         | Logistics        | Rs.  63,500.00 |
+--------+----------------------+------------------+----------------+



In [7]:
# Deleting employee record ID 103 and verifying roster
remove_employee(103)
list_employees()

  [SUCCESS] Successfully expunged Employee ID 103 from database.

+--------+----------------------+------------------+----------------+
| Emp ID | Employee Name        | Department       | Salary (INR)   |
+--------+----------------------+------------------+----------------+
| 101    | Pranjal Khadke       | Engineering      | Rs.  75,000.00 |
| 102    | Rohan Sharma         | Data Science     | Rs.  82,000.00 |
| 104    | Vikram Patel         | Logistics        | Rs.  63,500.00 |
+--------+----------------------+------------------+----------------+



In [8]:
# Integrity and exception testing
add_employee(101, "Duplicate Test", "HR", 40000)
find_employee(999)

  [ERROR] Primary Key Collision: Employee ID 101 already exists.
  [NOT FOUND] No employee matches ID: 999.


<div style="background-color: #f0fdf4; border: 1px solid #bbf7d0; border-left: 5px solid #16a34a; padding: 16px 20px; border-radius: 6px; font-family: sans-serif;">
  <h4 style="margin: 0 0 6px 0; color: #15803d; font-size: 15px; font-weight: 700;">Conclusion & Experimental Findings</h4>
  <p style="margin: 0; color: #166534; font-size: 13.5px; line-height: 1.6;">
    The flat CSV storage model provides lightweight, dependency-free persistence ideal for micro-scale utilities. Primary key integrity was successfully maintained through read-before-write validation. The trade-off observed is that update and delete operations operate at <strong>O(N) time complexity</strong> due to full file rewrites, highlighting why relational engines (like SQLite) are favored as dataset volumes scale.
  </p>
</div>